# 06 — Snapshots, provenance et calendriers maison

Ce notebook explique **d'où viennent les données**, pourquoi elles sont figées, comment
une correction amont arrive jusqu'à vous, et comment ajouter vos propres fermetures sans
forker quoi que ce soit.

In [1]:
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd

import better_calendar as bcal
from better_calendar.calendars import snapshot

## 1. Le problème que les snapshots résolvent

Si `bcal.get("XNYS")` interrogeait `exchange-calendars` au moment de la question, alors un
`pip install --upgrade` pourrait déplacer une date de règlement sans que personne ne le
décide. Le back-office recalcule, la date bouge d'un jour, personne ne l'a vu.

La parade : les données sont matérialisées une fois, **commitées**, embarquées dans la
wheel, et lues depuis le disque. Les fournisseurs ne sont même pas installés sur les
machines qui consomment la bibliothèque.

In [2]:
# Trois moments totalement distincts.
pd.DataFrame(
    [
        {"quand": "rarement, à la main", "quoi": "better-calendar snapshot : importe les 4 sources, écrit les fichiers", "qui": "vous"},
        {"quand": "au build de la wheel", "quoi": "les fichiers commités sont copiés — aucun calcul", "qui": "uv build"},
        {"quand": "à l'exécution", "quoi": "lecture d'un fichier local, ~0,06 ms", "qui": "l'utilisateur"},
    ]
).set_index("quand")

,quoi,qui
quand,,
"rarement, à la main",better-calendar snapshot : importe les 4 sourc...,vous
au build de la wheel,les fichiers commités sont copiés — aucun calcul,uv build
à l'exécution,"lecture d'un fichier local, ~0,06 ms",l'utilisateur


In [3]:
# Preuve : aucun fournisseur n'est importé, même après avoir tout interrogé.
code = (
    "import sys, better_calendar as bcal\n"
    "[bcal.get(n) for n in bcal.list()]\n"
    "print([m for m in ('exchange_calendars','holidays','QuantLib','workalendar') if m in sys.modules])\n"
)
print(subprocess.run([sys.executable, "-c", code], capture_output=True, text=True).stdout)

[]



## 2. Ce que contient le snapshot

In [4]:
manifeste = snapshot.load_manifest()
resume = pd.DataFrame(
    [{"source": e.provider, "version": e.provider_version} for e in manifeste.values()]
).value_counts().reset_index(name="calendriers")
resume

,source,version,calendriers
0,python_holidays,0.83,251
1,quantlib,1.43,91
2,workalendar,17.0.0,76
3,exchange_calendars,4.5.6,59


Chaque calendrier a une ligne de manifeste : d'où il vient, dans quelle version, sur quel
horizon, et une empreinte SHA-256 du fichier. C'est cette empreinte que la vérification de
dérive compare.

In [5]:
entree = manifeste["XNYS"]
pd.Series(entree.to_json())

provider                                           exchange_calendars
provider_version                                                4.5.6
upstream                                                         XNYS
bounds                                       [1970-01-01, 2100-12-31]
weekmask                                          Mon Tue Wed Thu Fri
tz                                                   America/New_York
session_start                                                00:00:00
holidays                                                         1227
sha256              0ecf92382cb246f246308d96394af3f2615d417e332076...
dtype: object

Le format est du **texte**, une date ISO par ligne, un fichier par calendrier. Ce choix
n'est pas anodin : tout le mécanisme de dérive repose sur le fait qu'un changement amont
arrive sous forme de *pull request lisible*. Un blob Parquet afficherait « fichier binaire
modifié » ; là, la PR montre les deux lignes qui bougent.

In [6]:
chemin = snapshot.DATA_DIR / "calendars" / entree.filename
print(f"{chemin.name} — {entree.holidays} lignes\n")
print("\n".join(chemin.read_text().splitlines()[:6]))
print("…")

XNYS.csv — 1227 lignes

1970-01-01
1970-02-23
1970-03-27
1970-07-03
1970-09-07
1970-11-26
…


## 3. Bornes honnêtes

Les bornes reflètent ce que la source sait **réellement** répondre, pas ce qu'on lui a
demandé. Deux causes distinctes :

- l'amont refuse explicitement (Tokyo avant 1997, Hong Kong après 2049) ;
- l'amont *dégrade silencieusement* — les fêtes lunaires, hébraïques et islamiques sont
  tabulées, et les tables s'arrêtent sans le dire.

In [7]:
from datetime import date

clippes = [(n, e) for n, e in manifeste.items() if e.bounds[1] != date(2100, 12, 31)]
print(f"{len(clippes)} calendriers sur {len(manifeste)} ont un horizon rétréci\n")
pd.DataFrame(
    [{"calendrier": n, "source": e.provider, "début": e.bounds[0], "fin": e.bounds[1]}
     for n, e in sorted(clippes)[:12]]
).set_index("calendrier")

62 calendriers sur 477 ont un horizon rétréci



,source,début,fin
calendrier,,,
XBOM,exchange_calendars,1997-01-01,2024-12-31
XHKG,exchange_calendars,1970-01-01,2049-12-31
XKRX,exchange_calendars,1970-01-01,2050-12-31
XSAU,exchange_calendars,2021-01-01,2024-12-31
XSES,exchange_calendars,1986-01-01,2024-12-31
XSHG,exchange_calendars,1990-12-03,2025-12-31
country:AE,python_holidays,1972-01-01,2077-12-31
country:BH,python_holidays,1972-01-01,2076-12-31
country:BN,python_holidays,1984-01-01,2077-12-31


Le second cas est le plus vicieux. Au-delà de 2026, le calendrier de Shanghai de QuantLib
renvoie **un** jour férié par an au lieu de dix-huit — le Nouvel An chinois a disparu — et
continue de répondre « oui, jour ouvré » avec aplomb.

La génération détecte cet effondrement de densité et coupe l'horizon, donc on obtient une
erreur au lieu d'une mauvaise réponse :

In [8]:
chine = bcal.get("ql:China.SSE")
print("bornes :", chine.bounds)
try:
    chine.is_bday("2030-02-14")
except bcal.OutOfBoundsError as exc:
    print("\n" + str(exc))

bornes : (datetime.date(1970, 1, 1), datetime.date(2026, 12, 31))

2030-02-14 is outside the bounds of calendar 'ql:China.SSE' (1970-01-01 to 2026-12-31, inclusive). Rebuild the calendar with wider `bounds`, or raise MAX_YEAR in better_calendar.core.epoch.


## 4. Comment une correction amont vous parvient

```
exchange-calendars publie 4.6.0
        ↓
job CI hebdomadaire : installe les dernières versions,
régénère en mémoire, compare aux fichiers commités
        ↓
différence détectée → sortie non nulle → une PR est ouverte
        ↓
    vous lisez la PR :   XNYS
                         + 2027-05-31
                         - 2027-06-01
        ↓
vous mergez (ou pas) → nouvelle version de better-calendar
```

Le point clé : le changement est une PR que quelqu'un **lit et approuve**, pas un effet de
bord d'un `pip upgrade`.

In [9]:
# La ligne de commande qui pilote tout ça.
for commande in (
    ["describe", "rate:SOFR"],
    ["next", "XNYS", "2026-07-31", "+5"],
    ["list", "--provider", "builtin"],
):
    sortie = subprocess.run(
        [sys.executable, "-m", "better_calendar.cli", *commande],
        capture_output=True, text=True,
    )
    print(f"$ better-calendar {' '.join(commande)}")
    print("\n".join(sortie.stdout.splitlines()[:8]))
    print()

$ better-calendar describe rate:SOFR
{
  "name": "rate:SOFR",
  "weekmask": "Mon Tue Wed Thu Fri",
  "bounds": [
    "1970-01-01",
    "2100-12-31"
  ],
  "tz": null,

$ better-calendar next XNYS 2026-07-31 +5
2026-08-07



$ better-calendar list --provider builtin
crypto:24x7
weekday



`better-calendar diff` régénère tout en mémoire et compare. Sortie non nulle si une date a
bougé. C'est ce que le job hebdomadaire exécute — ici on le lance sur trois calendriers
pour que ce soit rapide (il faut les extras fournisseurs installés).

In [10]:
sortie = subprocess.run(
    [sys.executable, "-m", "better_calendar.cli", "diff",
     "--provider", "quantlib", "--only", "fin:TARGET2,rate:SOFR,fin:LNB"],
    capture_output=True, text=True,
)
print("code de sortie :", sortie.returncode)
print(sortie.stdout or sortie.stderr.splitlines()[-1])

code de sortie : 0
snapshot is current (3 calendars, versions {'quantlib': '1.43'})



## 5. Vos propres calendriers

Un desk ferme le 24 décembre ; la zone euro non. C'est un fait local, pas une erreur de
QuantLib — et la réponse n'est **jamais** de forker un calendrier fournisseur, parce qu'un
fork cesse silencieusement de recevoir les corrections amont.

### Option A — le fichier de configuration

Trouvé via `$BETTER_CALENDAR_CONFIG`, ou `./better-calendar.yaml` dans le répertoire
courant. Le TOML marche à l'identique.

In [11]:
import os
import tempfile

from better_calendar.calendars.registry import reload_config

CONFIG = """
calendars:
  desk:paris:
    base: fin:TARGET2
    extra_holidays: ["2026-12-24", "2026-12-31"]
    tz: Europe/Paris

  desk:gulf:
    weekmask: "Sun Mon Tue Wed Thu"
    bounds: ["2020-01-01", "2035-12-31"]
    tz: Asia/Dubai
"""

dossier = Path(tempfile.mkdtemp())
(dossier / "better-calendar.yaml").write_text(CONFIG)
os.environ["BETTER_CALENDAR_CONFIG"] = str(dossier / "better-calendar.yaml")
reload_config()

desk = bcal.get("desk:paris")
base = bcal.get("fin:TARGET2")
pd.DataFrame(
    [
        {"date": d,
         "fin:TARGET2": base.is_bday(d),
         "desk:paris": desk.is_bday(d)}
        for d in ("2026-04-06", "2026-12-24", "2026-12-25", "2026-12-31")
    ]
).set_index("date")

,fin:TARGET2,desk:paris
date,,
2026-04-06,False,False
2026-12-24,True,False
2026-12-25,False,False
2026-12-31,True,False


Le desk hérite de **tout** ce que TARGET2 ferme (lundi de Pâques), plus ses propres jours.
Et la base reste intacte.

In [12]:
print("desk:gulf weekmask :", bcal.get("desk:gulf").weekmask)
print("dimanche ouvré ?   :", bcal.get("desk:gulf").is_bday("2026-08-02"))
print("provenance         :", bcal.describe("desk:paris")["provider"],
      "/", bcal.describe("desk:paris")["provider_version"])

desk:gulf weekmask : Mon Tue Wed Thu Sun
dimanche ouvré ?   : True
provenance         : custom / fin:TARGET2


Nommer une entrée comme un calendrier livré le **masque**, donc les call sites existants
récupèrent la version locale sans changer une ligne :

In [13]:
(dossier / "better-calendar.yaml").write_text("""
calendars:
  XNYS:
    base: XNYS
    extra_holidays: ["2026-11-27"]
""")
reload_config()

print("lendemain de Thanksgiving ouvré au NYSE ?", bcal.get("XNYS").is_bday("2026-11-27"))
print("via l'alias NYSE aussi                  ?", bcal.get("NYSE").is_bday("2026-11-27"))

lendemain de Thanksgiving ouvré au NYSE ? False
via l'alias NYSE aussi                  ? False


In [14]:
del os.environ["BETTER_CALENDAR_CONFIG"]
reload_config()
print("configuration retirée, XNYS revient au snapshot :", bcal.get("XNYS").is_bday("2026-11-27"))

configuration retirée, XNYS revient au snapshot : True


### Option B — par programme

In [15]:
from better_calendar import Calendar

sur_mesure = bcal.get("XNYS").with_holidays(["2026-11-27"], name="desk:us")
bcal.register("desk:us", sur_mesure)

print("desk:us  :", bcal.get("desk:us").is_bday("2026-11-27"))
print("XNYS     :", bcal.get("XNYS").is_bday("2026-11-27"))
print("listé    :", "desk:us" in bcal.list())

bcal.unregister("desk:us")

desk:us  : False
XNYS     : True
listé    : True


### Ordre de résolution

1. déjà un `Calendar` — passe tel quel ;
2. enregistré via `register()` ;
3. défini dans le fichier de configuration ;
4. intégré (`weekday`, `crypto:24x7`) ;
5. snapshot commité ;
6. alias, puis on recommence à l'étape 2.

## Récapitulatif

| Appel / commande | Rôle |
|---|---|
| `snapshot.load_manifest()` | l'index : source, version, bornes, empreinte |
| `bcal.describe(nom)` | la même chose pour un calendrier |
| `better-calendar snapshot` | régénérer depuis les sources amont |
| `better-calendar diff` | détecter une dérive, sortie non nulle |
| `better-calendar describe` / `next` / `list` | inspection en ligne de commande |
| `./better-calendar.yaml` | calendriers d'organisation, composés |
| `register()` / `unregister()` | la même chose par programme |

**Retour au début :** [01 — Prise en main](01-prise-en-main.ipynb)